In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve
)
# 2. CHARGEMENT DES DONNÉES

df = pd.read_csv("P02_scoring_credit_madina.csv")

df.head()

print("Dimensions :", df.shape)
print(df.head())

# 3. COMPRÉHENSION DES DONNÉES


print("\nInformations générales :")
print(df.info())

print("\nStatistiques descriptives :")
print(df.describe(include="all"))

# Valeurs manquantes
missing = df.isnull().sum()
print("\nValeurs manquantes :")
print(missing)

# Pourcentage de valeurs manquantes
missing_pct = (missing / len(df)) * 100
print("\nPourcentage de valeurs manquantes :")
print(missing_pct)

# 4. ANALYSE EXPLORATOIRE


# Détection automatique de la variable cible
possible_targets = ["CreditRisk", "Risk", "credit_risk", "class", "Class"]
target = None
for c in possible_targets:
    if c in df.columns:
        target = c
        break

if target is None:
    target = df.columns[-1]
    print(f"Variable cible supposée : {target}")

# Distribution de la variable cible
plt.figure(figsize=(5,4))
sns.countplot(x=target, data=df)
plt.title("Répartition de la variable cible")
plt.show()

# Histogrammes des variables numériques
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    plt.figure(figsize=(5,3))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution de {col}")
    plt.tight_layout()
    plt.show()

# Matrice de corrélation
if len(num_cols) > 1:
    plt.figure(figsize=(10,8))
    sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm")
    plt.title("Matrice de corrélation")
    plt.show()


# 5. PRÉPARATION DES DONNÉES
# Suppression des doublons
df = df.drop_duplicates()

# Encodage de la variable cible
if df[target].dtype == "object":
    y = df[target].map({
        "good":1, "Good":1, "1":1, 1:1,
        "bad":0, "Bad":0, "2":0, 2:0
    })
    if y.isnull().any():
        y = pd.factorize(df[target])[0]
else:
    y = df[target]

X = df.drop(columns=[target])

# Variables numériques et catégorielles
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Prétraitement
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Séparation train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# 6. MODÉLISATION

models = {
    "Régression Logistique": LogisticRegression(max_iter=1000),
    "Arbre de Décision": DecisionTreeClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

best_model = None
best_auc = -1

for name, model in models.items():

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("clf", model)
    ])

    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["clf"], "predict_proba"):
        proba = pipeline.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, proba)
    else:
        auc = np.nan

    results.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Précision": precision_score(y_test, pred),
        "Rappel": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": auc
    })

    if auc > best_auc:
        best_auc = auc
        best_model = pipeline

# Tableau comparatif
results_df = pd.DataFrame(results)
print("\nComparaison des modèles :")
print(results_df.sort_values(by="ROC-AUC", ascending=False))


# 7. ÉVALUATION DU MEILLEUR MODÈLE


best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:,1]

print("\nRapport de classification :")
print(classification_report(y_test, best_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues")
plt.title("Matrice de confusion")
plt.show()

# Courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, best_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {best_auc:.3f}")
plt.plot([0,1], [0,1], "--", color="gray")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC")
plt.legend()
plt.show()


# 8. CHOIX DU SEUIL DE DÉCISION


# Seuil classique
threshold_default = 0.50
pred_default = (best_proba >= threshold_default).astype(int)

# Seuil plus prudent pour une microfinance
threshold_business = 0.30
pred_business = (best_proba >= threshold_business).astype(int)

print("\n=== Comparaison des seuils ===")

for t, pred in [(0.50, pred_default), (0.30, pred_business)]:

    print(f"\nSeuil = {t}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Précision :", precision_score(y_test, pred))
    print("Rappel :", recall_score(y_test, pred))
    print("F1 :", f1_score(y_test, pred))

# 9. CONCLUSION

print("\nLe meilleur modèle est :")
print(results_df.sort_values(by="ROC-AUC", ascending=False).iloc[0])



2026-08-14T00:02:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-14T00:02:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-14T00:02:40Z INF +--------------------------------------------------------------------------------------------+
2026-08-14T00:02:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-14T00:02:40Z INF |  https://encryption-custom-pole-toolbar.trycloudflare.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')